# UTCI Sulawesi 1996–2025 — STEP 3
## Klasifikasi Thermal Stress + Statistik Tahunan

Notebook ini dijalankan setelah **STEP 2 dinyatakan LULUS**.

### Input
`D:\MERLI\data\UTCI_Hasil\03_utci_dailymax_1996_2025_C_sulawesi.nc`

Variabel:
- `utci_max`
- unit: °C
- temporal: daily maximum
- periode: 1996–2025
- grid: mask Sulawesi yang sama dengan Tmax

### Skala UTCI yang digunakan

Klasifikasi mengikuti skala UTCI standar yang dirujuk pada Błażejczyk et al. (2013):

| Kode | UTCI (°C) | Kategori |
|---:|---:|---|
| 1 | < -40 | Extreme cold stress |
| 2 | -40 sampai < -27 | Very strong cold stress |
| 3 | -27 sampai < -13 | Strong cold stress |
| 4 | -13 sampai < 0 | Moderate cold stress |
| 5 | 0 sampai < 9 | Slight cold stress |
| 6 | 9 sampai < 26 | No thermal stress |
| 7 | 26 sampai < 32 | Moderate heat stress |
| 8 | 32 sampai < 38 | Strong heat stress |
| 9 | 38 sampai 46 | Very strong heat stress |
| 10 | > 46 | Extreme heat stress |

Karena data pada STEP 2 memiliki rentang sekitar **18.57–44.47 °C**, secara praktis Sulawesi pada dataset daily maximum ini diperkirakan hanya muncul pada kategori:
- No thermal stress;
- Moderate heat stress;
- Strong heat stress;
- Very strong heat stress.

Tetapi kode tetap menyimpan **seluruh 10 kategori** agar metode lengkap dan tidak bergantung pada asumsi rentang data.

### Output utama
1. `05_utci_thermal_stress_class_daily_1996_2025.nc`
2. `06_utci_thermal_stress_annual_1996_2025.nc`
3. `07_utci_thermal_stress_annual_spatial_mean.csv`

Tahap ini **belum melakukan trend analysis atau plotting**.

In [1]:
# =========================================================
# 0. IMPORT
# =========================================================

from pathlib import Path
import gc

import numpy as np
import pandas as pd
import xarray as xr

print("xarray :", xr.__version__)
print("pandas :", pd.__version__)
print("numpy  :", np.__version__)

xarray : 2025.10.1
pandas : 2.3.3
numpy  : 2.3.5


In [2]:
# =========================================================
# 1. KONFIGURASI
# =========================================================

RESULT_DIR = Path(
    r"D:\MERLI\data\UTCI_Hasil"
)

INPUT_FILE = (
    RESULT_DIR
    / "03_utci_dailymax_1996_2025_C_sulawesi.nc"
)

DAILY_CLASS_FILE = (
    RESULT_DIR
    / "05_utci_thermal_stress_class_daily_1996_2025.nc"
)

ANNUAL_FILE = (
    RESULT_DIR
    / "06_utci_thermal_stress_annual_1996_2025.nc"
)

SUMMARY_FILE = (
    RESULT_DIR
    / "07_utci_thermal_stress_annual_spatial_mean.csv"
)

START_YEAR = 1996
END_YEAR = 2025

FILL_CLASS = np.int8(-127)

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Input STEP 2 tidak ditemukan:\n{INPUT_FILE}"
    )

print("INPUT :", INPUT_FILE)
print("OUTPUT daily :", DAILY_CLASS_FILE)
print("OUTPUT annual:", ANNUAL_FILE)
print("OUTPUT CSV   :", SUMMARY_FILE)

INPUT : D:\MERLI\data\UTCI_Hasil\03_utci_dailymax_1996_2025_C_sulawesi.nc
OUTPUT daily : D:\MERLI\data\UTCI_Hasil\05_utci_thermal_stress_class_daily_1996_2025.nc
OUTPUT annual: D:\MERLI\data\UTCI_Hasil\06_utci_thermal_stress_annual_1996_2025.nc
OUTPUT CSV   : D:\MERLI\data\UTCI_Hasil\07_utci_thermal_stress_annual_spatial_mean.csv


## 2. Buka dan QC cepat input STEP 2

Kita pastikan lagi dataset yang masuk adalah daily maximum UTCI dalam °C.

In [3]:
# =========================================================
# 2. LOAD INPUT
# =========================================================

ds = xr.open_dataset(
    INPUT_FILE,
    engine="netcdf4",
    decode_times=True,
    decode_timedelta=False
)

if "utci_max" not in ds.data_vars:
    raise ValueError(
        "Variabel 'utci_max' tidak ditemukan."
    )

utci = ds["utci_max"]

required_dims = {
    "time",
    "latitude",
    "longitude"
}

if set(utci.dims) != required_dims:
    utci = utci.transpose(
        "time",
        "latitude",
        "longitude"
    )

dates = pd.DatetimeIndex(
    utci["time"].values
)

print("Shape     :", utci.shape)
print("Units     :", utci.attrs.get("units", "-"))
print("Awal      :", dates.min())
print("Akhir     :", dates.max())
print("N hari    :", len(dates))
print("Min UTCI  :", float(utci.min(skipna=True).values), "°C")
print("Max UTCI  :", float(utci.max(skipna=True).values), "°C")

expected_dates = pd.date_range(
    f"{START_YEAR}-01-01",
    f"{END_YEAR}-12-31",
    freq="D"
)

if len(dates) != len(expected_dates):
    raise ValueError(
        "Jumlah hari input tidak sesuai periode 1996–2025."
    )

if len(dates[dates.duplicated()]) != 0:
    raise ValueError(
        "Masih terdapat tanggal duplikat."
    )

if len(expected_dates.difference(dates)) != 0:
    raise ValueError(
        "Masih terdapat tanggal hilang."
    )

print("\nINPUT STEP 2: LULUS")

Shape     : (10958, 44, 32)
Units     : degC
Awal      : 1996-01-01 00:00:00
Akhir     : 2025-12-31 00:00:00
N hari    : 10958
Min UTCI  : 18.56707763671875 °C
Max UTCI  : 44.473175048828125 °C

INPUT STEP 2: LULUS


## 3. Muat data ke memori

Ukuran dataset ini masih aman untuk diproses langsung:
`10958 × 44 × 32`.

Nilai di luar mask Sulawesi tetap `NaN`.

In [4]:
# =========================================================
# 3. LOAD ARRAY
# =========================================================

values = np.asarray(
    utci.values,
    dtype="float32"
)

valid = np.isfinite(values)

valid_grid = np.any(
    valid,
    axis=0
)

print("Shape array   :", values.shape)
print("Grid valid    :", int(valid_grid.sum()))
print("Expected mask :", 242)

if int(valid_grid.sum()) != 242:
    print(
        "PERINGATAN: grid valid bukan 242. "
        "Periksa kembali mask jika angka ini tidak disengaja."
    )

print(
    "Total nilai valid:",
    int(valid.sum())
)

Shape array   : (10958, 44, 32)
Grid valid    : 242
Expected mask : 242
Total nilai valid: 2651836


## 4. Klasifikasi harian UTCI

Setiap nilai `utci_max` diubah menjadi kode kategori thermal stress.

Kode `-127` digunakan sebagai FillValue untuk area di luar Sulawesi.

In [5]:
# =========================================================
# 4. KLASIFIKASI UTCI HARIAN
# =========================================================

stress_class = np.full(
    values.shape,
    FILL_CLASS,
    dtype="int8"
)

u = values

# Cold stress
stress_class[
    valid & (u < -40.0)
] = 1

stress_class[
    valid & (u >= -40.0) & (u < -27.0)
] = 2

stress_class[
    valid & (u >= -27.0) & (u < -13.0)
] = 3

stress_class[
    valid & (u >= -13.0) & (u < 0.0)
] = 4

stress_class[
    valid & (u >= 0.0) & (u < 9.0)
] = 5

# No thermal stress
stress_class[
    valid & (u >= 9.0) & (u < 26.0)
] = 6

# Heat stress
stress_class[
    valid & (u >= 26.0) & (u < 32.0)
] = 7

stress_class[
    valid & (u >= 32.0) & (u < 38.0)
] = 8

stress_class[
    valid & (u >= 38.0) & (u <= 46.0)
] = 9

stress_class[
    valid & (u > 46.0)
] = 10


# =========================================================
# QC: SEMUA NILAI VALID HARUS TERKLASIFIKASI
# =========================================================

classified_valid = (
    stress_class != FILL_CLASS
)

unclassified = (
    valid
    & ~classified_valid
)

print(
    "Nilai valid       :",
    int(valid.sum())
)

print(
    "Nilai classified  :",
    int((classified_valid & valid).sum())
)

print(
    "Unclassified      :",
    int(unclassified.sum())
)

if int(unclassified.sum()) != 0:
    raise RuntimeError(
        "Ada nilai UTCI valid yang tidak masuk kategori."
    )

print("KLASIFIKASI HARIAN: LULUS")

Nilai valid       : 2651836
Nilai classified  : 2651836
Unclassified      : 0
KLASIFIKASI HARIAN: LULUS


## 5. Ringkasan seluruh periode

Bagian ini hanya untuk QC awal: berapa banyak grid-day yang masuk tiap kategori selama 1996–2025.

In [6]:
# =========================================================
# 5. DISTRIBUSI KATEGORI SELURUH PERIODE
# =========================================================

CATEGORY_NAMES = {
    1: "Extreme cold stress",
    2: "Very strong cold stress",
    3: "Strong cold stress",
    4: "Moderate cold stress",
    5: "Slight cold stress",
    6: "No thermal stress",
    7: "Moderate heat stress",
    8: "Strong heat stress",
    9: "Very strong heat stress",
    10: "Extreme heat stress",
}

print("========== DISTRIBUSI 1996–2025 ==========")

total_valid = int(valid.sum())

for code_value, name in CATEGORY_NAMES.items():

    n = int(
        np.sum(
            stress_class == code_value
        )
    )

    pct = (
        100.0 * n / total_valid
        if total_valid > 0
        else np.nan
    )

    print(
        f"{code_value:2d} | "
        f"{name:25s} | "
        f"{n:10d} | "
        f"{pct:7.3f}%"
    )

========== DISTRIBUSI 1996–2025 ==========
 1 | Extreme cold stress       |          0 |   0.000%
 2 | Very strong cold stress   |          0 |   0.000%
 3 | Strong cold stress        |          0 |   0.000%
 4 | Moderate cold stress      |          0 |   0.000%
 5 | Slight cold stress        |          0 |   0.000%
 6 | No thermal stress         |      13940 |   0.526%
 7 | Moderate heat stress      |     711490 |  26.830%
 8 | Strong heat stress        |    1870627 |  70.541%
 9 | Very strong heat stress   |      55779 |   2.103%
10 | Extreme heat stress       |          0 |   0.000%


## 6. Simpan klasifikasi harian

File ini berguna kalau nanti kita ingin:
- melihat kategori UTCI pada tanggal tertentu;
- membandingkan hari heatwave Tmax dengan kategori UTCI;
- membuat peta frekuensi thermal stress.

In [7]:
# =========================================================
# 6. SAVE DAILY CLASSIFICATION
# =========================================================

daily_class_da = xr.DataArray(
    stress_class,
    coords={
        "time": ds["time"].values,
        "latitude": ds["latitude"].values,
        "longitude": ds["longitude"].values,
    },
    dims=(
        "time",
        "latitude",
        "longitude"
    ),
    name="utci_stress_class",
)

daily_class_da.attrs = {
    "long_name": (
        "UTCI thermal stress category based on daily maximum UTCI"
    ),
    "flag_values": np.arange(
        1,
        11,
        dtype="int8"
    ),
    "flag_meanings": (
        "extreme_cold_stress "
        "very_strong_cold_stress "
        "strong_cold_stress "
        "moderate_cold_stress "
        "slight_cold_stress "
        "no_thermal_stress "
        "moderate_heat_stress "
        "strong_heat_stress "
        "very_strong_heat_stress "
        "extreme_heat_stress"
    ),
    "classification": (
        "<-40 ECS; "
        "-40 to <-27 VSCS; "
        "-27 to <-13 SCS; "
        "-13 to <0 MCS; "
        "0 to <9 SLCS; "
        "9 to <26 NTS; "
        "26 to <32 MHS; "
        "32 to <38 SHS; "
        "38 to 46 VSHS; "
        ">46 EHS"
    ),
    "source_index": "UTCI daily maximum",
    "reference": "Błażejczyk et al. (2013)",
}

daily_class_ds = daily_class_da.to_dataset()

daily_class_ds.attrs = {
    "title": (
        "Daily UTCI Thermal Stress Classification over Sulawesi"
    ),
    "period": "1996-2025",
    "temporal_basis": "daily maximum UTCI",
}

encoding_daily = {
    "utci_stress_class": {
        "dtype": "int8",
        "_FillValue": int(FILL_CLASS),
        "zlib": True,
        "complevel": 4,
        "shuffle": True,
    }
}

if DAILY_CLASS_FILE.exists():
    DAILY_CLASS_FILE.unlink()

daily_class_ds.to_netcdf(
    DAILY_CLASS_FILE,
    engine="netcdf4",
    encoding=encoding_daily
)

print("Tersimpan:")
print(DAILY_CLASS_FILE)

Tersimpan:
D:\MERLI\data\UTCI_Hasil\05_utci_thermal_stress_class_daily_1996_2025.nc


## 7. Statistik tahunan per grid

Untuk setiap tahun dan setiap grid Sulawesi dihitung:

- `NTS_days` = jumlah hari no thermal stress;
- `MHS_days` = jumlah hari moderate heat stress;
- `SHS_days` = jumlah hari strong heat stress;
- `VSHS_days` = jumlah hari very strong heat stress;
- `EHS_days` = jumlah hari extreme heat stress;
- `HeatStress_days` = jumlah hari UTCI ≥ 26 °C;
- `StrongPlus_days` = jumlah hari UTCI ≥ 32 °C;
- `VeryStrongPlus_days` = jumlah hari UTCI ≥ 38 °C;
- `UTCI_mean` = rata-rata daily maximum UTCI dalam tahun tersebut;
- `UTCI_max` = daily maximum UTCI tertinggi dalam tahun tersebut.

**Catatan:** kategori heat stress bersifat saling eksklusif, sedangkan `HeatStress_days`, `StrongPlus_days`, dan `VeryStrongPlus_days` adalah indikator kumulatif.

In [8]:
# =========================================================
# 7. STATISTIK TAHUNAN
# =========================================================

years = np.arange(
    START_YEAR,
    END_YEAR + 1,
    dtype=int
)

ny = len(years)
nlat = values.shape[1]
nlon = values.shape[2]

annual_vars = {
    "NTS_days": np.full(
        (ny, nlat, nlon),
        np.nan,
        dtype="float32"
    ),
    "MHS_days": np.full(
        (ny, nlat, nlon),
        np.nan,
        dtype="float32"
    ),
    "SHS_days": np.full(
        (ny, nlat, nlon),
        np.nan,
        dtype="float32"
    ),
    "VSHS_days": np.full(
        (ny, nlat, nlon),
        np.nan,
        dtype="float32"
    ),
    "EHS_days": np.full(
        (ny, nlat, nlon),
        np.nan,
        dtype="float32"
    ),
    "HeatStress_days": np.full(
        (ny, nlat, nlon),
        np.nan,
        dtype="float32"
    ),
    "StrongPlus_days": np.full(
        (ny, nlat, nlon),
        np.nan,
        dtype="float32"
    ),
    "VeryStrongPlus_days": np.full(
        (ny, nlat, nlon),
        np.nan,
        dtype="float32"
    ),
    "UTCI_mean": np.full(
        (ny, nlat, nlon),
        np.nan,
        dtype="float32"
    ),
    "UTCI_max": np.full(
        (ny, nlat, nlon),
        np.nan,
        dtype="float32"
    ),
}

date_years = dates.year.values

for yi, year in enumerate(years):

    idx = np.where(
        date_years == year
    )[0]

    expected_n = (
        366
        if pd.Timestamp(
            f"{year}-12-31"
        ).is_leap_year
        else 365
    )

    if len(idx) != expected_n:
        raise ValueError(
            f"{year}: jumlah hari {len(idx)}, "
            f"seharusnya {expected_n}."
        )

    cls_y = stress_class[
        idx,
        :,
        :
    ]

    val_y = values[
        idx,
        :,
        :
    ]

    valid_y = np.isfinite(
        val_y
    )

    valid_grid_y = np.any(
        valid_y,
        axis=0
    )

    # kategori eksklusif
    annual_vars["NTS_days"][yi] = np.sum(
        cls_y == 6,
        axis=0
    )

    annual_vars["MHS_days"][yi] = np.sum(
        cls_y == 7,
        axis=0
    )

    annual_vars["SHS_days"][yi] = np.sum(
        cls_y == 8,
        axis=0
    )

    annual_vars["VSHS_days"][yi] = np.sum(
        cls_y == 9,
        axis=0
    )

    annual_vars["EHS_days"][yi] = np.sum(
        cls_y == 10,
        axis=0
    )

    # indikator kumulatif
    annual_vars["HeatStress_days"][yi] = np.sum(
        valid_y & (val_y >= 26.0),
        axis=0
    )

    annual_vars["StrongPlus_days"][yi] = np.sum(
        valid_y & (val_y >= 32.0),
        axis=0
    )

    annual_vars["VeryStrongPlus_days"][yi] = np.sum(
        valid_y & (val_y >= 38.0),
        axis=0
    )

    # statistik UTCI
    with np.errstate(
        invalid="ignore"
    ):
        annual_vars["UTCI_mean"][yi] = np.nanmean(
            val_y,
            axis=0
        )

        annual_vars["UTCI_max"][yi] = np.nanmax(
            val_y,
            axis=0
        )

    # luar mask dikembalikan ke NaN
    for name in annual_vars:
        annual_vars[name][
            yi,
            ~valid_grid_y
        ] = np.nan

    print(
        f"{year}: {len(idx)} hari selesai"
    )

print("\nSTATISTIK TAHUNAN: SELESAI")

1996: 366 hari selesai
1997: 365 hari selesai
1998: 365 hari selesai
1999: 365 hari selesai
2000: 366 hari selesai
2001: 365 hari selesai
2002: 365 hari selesai
2003: 365 hari selesai
2004: 366 hari selesai
2005: 365 hari selesai
2006: 365 hari selesai
2007: 365 hari selesai
2008: 366 hari selesai
2009: 365 hari selesai
2010: 365 hari selesai
2011: 365 hari selesai
2012: 366 hari selesai
2013: 365 hari selesai
2014: 365 hari selesai
2015: 365 hari selesai
2016: 366 hari selesai
2017: 365 hari selesai
2018: 365 hari selesai
2019: 365 hari selesai
2020: 366 hari selesai
2021: 365 hari selesai
2022: 365 hari selesai
2023: 365 hari selesai
2024: 366 hari selesai
2025: 365 hari selesai

STATISTIK TAHUNAN: SELESAI


C:\Users\User\AppData\Local\Temp\ipykernel_17652\3429292400.py:157: RuntimeWarning: Mean of empty slice
  annual_vars["UTCI_mean"][yi] = np.nanmean(
C:\Users\User\AppData\Local\Temp\ipykernel_17652\3429292400.py:162: RuntimeWarning: All-NaN slice encountered
  annual_vars["UTCI_max"][yi] = np.nanmax(


## 8. QC konsistensi kategori tahunan

Untuk setiap grid valid:

`semua kategori thermal stress = jumlah hari valid dalam tahun`

Karena rentang data Sulawesi pada STEP 2 > 9 °C, secara praktis jumlah:

`NTS + MHS + SHS + VSHS + EHS`

seharusnya sama dengan jumlah hari valid pada tiap tahun.

In [9]:
# =========================================================
# 8. QC KATEGORI TAHUNAN
# =========================================================

qc_fail = 0

for yi, year in enumerate(years):

    idx = np.where(
        date_years == year
    )[0]

    valid_days = np.sum(
        np.isfinite(
            values[
                idx,
                :,
                :
            ]
        ),
        axis=0
    )

    # Karena input minimum STEP 2 > 9 °C,
    # hanya kelas 6–10 yang seharusnya muncul.
    category_sum = (
        annual_vars["NTS_days"][yi]
        + annual_vars["MHS_days"][yi]
        + annual_vars["SHS_days"][yi]
        + annual_vars["VSHS_days"][yi]
        + annual_vars["EHS_days"][yi]
    )

    mask = valid_grid

    mismatch = np.sum(
        ~np.isclose(
            category_sum[mask],
            valid_days[mask],
            equal_nan=False
        )
    )

    if mismatch > 0:
        qc_fail += int(
            mismatch
        )

        print(
            f"{year}: mismatch pada {mismatch} grid"
        )

print(
    "Total mismatch:",
    qc_fail
)

if qc_fail != 0:
    raise RuntimeError(
        "Jumlah kategori tahunan tidak sama dengan jumlah hari valid."
    )

print("QC KATEGORI TAHUNAN: LULUS")

Total mismatch: 0
QC KATEGORI TAHUNAN: LULUS


## 9. Simpan statistik tahunan ke NetCDF

In [10]:
# =========================================================
# 9. SAVE ANNUAL DATASET
# =========================================================

coords = {
    "year": years,
    "latitude": ds["latitude"].values,
    "longitude": ds["longitude"].values,
}

annual_ds = xr.Dataset(
    {
        name: (
            (
                "year",
                "latitude",
                "longitude"
            ),
            array
        )
        for name, array
        in annual_vars.items()
    },
    coords=coords
)

for name in [
    "NTS_days",
    "MHS_days",
    "SHS_days",
    "VSHS_days",
    "EHS_days",
    "HeatStress_days",
    "StrongPlus_days",
    "VeryStrongPlus_days",
]:
    annual_ds[name].attrs["units"] = "days"

annual_ds["NTS_days"].attrs["long_name"] = (
    "annual number of no thermal stress days"
)

annual_ds["MHS_days"].attrs["long_name"] = (
    "annual number of moderate heat stress days"
)

annual_ds["SHS_days"].attrs["long_name"] = (
    "annual number of strong heat stress days"
)

annual_ds["VSHS_days"].attrs["long_name"] = (
    "annual number of very strong heat stress days"
)

annual_ds["EHS_days"].attrs["long_name"] = (
    "annual number of extreme heat stress days"
)

annual_ds["HeatStress_days"].attrs["long_name"] = (
    "annual number of days with daily maximum UTCI >= 26 degC"
)

annual_ds["StrongPlus_days"].attrs["long_name"] = (
    "annual number of days with daily maximum UTCI >= 32 degC"
)

annual_ds["VeryStrongPlus_days"].attrs["long_name"] = (
    "annual number of days with daily maximum UTCI >= 38 degC"
)

annual_ds["UTCI_mean"].attrs = {
    "long_name": (
        "annual mean of daily maximum UTCI"
    ),
    "units": "degC",
}

annual_ds["UTCI_max"].attrs = {
    "long_name": (
        "annual maximum of daily maximum UTCI"
    ),
    "units": "degC",
}

annual_ds.attrs = {
    "title": (
        "Annual UTCI Thermal Stress Statistics over Sulawesi"
    ),
    "period": "1996-2025",
    "source": (
        "daily maximum UTCI from STEP 2"
    ),
    "classification_reference": (
        "Błażejczyk et al. (2013)"
    ),
}

encoding_annual = {
    name: {
        "dtype": "float32",
        "_FillValue": -9999.0,
        "zlib": True,
        "complevel": 4,
        "shuffle": True,
    }
    for name in annual_ds.data_vars
}

if ANNUAL_FILE.exists():
    ANNUAL_FILE.unlink()

annual_ds.to_netcdf(
    ANNUAL_FILE,
    engine="netcdf4",
    encoding=encoding_annual
)

print("Tersimpan:")
print(ANNUAL_FILE)

Tersimpan:
D:\MERLI\data\UTCI_Hasil\06_utci_thermal_stress_annual_1996_2025.nc


## 10. Ringkasan rata-rata spasial Sulawesi per tahun

CSV ini memudahkan pengecekan pola tahunan sebelum masuk plotting dan trend analysis.

**Catatan:** rata-rata spasial dihitung dari grid Sulawesi yang valid, sama seperti pendekatan ringkasan Tmax sebelumnya.

In [11]:
# =========================================================
# 10. SUMMARY REGIONAL TAHUNAN
# =========================================================

summary = {
    "year": years
}

for name in annual_ds.data_vars:

    regional_mean = annual_ds[
        name
    ].mean(
        dim=[
            "latitude",
            "longitude"
        ],
        skipna=True
    )

    summary[
        name
    ] = np.asarray(
        regional_mean.values,
        dtype=float
    )

summary_df = pd.DataFrame(
    summary
)

summary_df.to_csv(
    SUMMARY_FILE,
    index=False
)

print(
    summary_df.to_string(
        index=False
    )
)

print("\nTersimpan:")
print(SUMMARY_FILE)

 year  NTS_days   MHS_days   SHS_days  VSHS_days  EHS_days  HeatStress_days  StrongPlus_days  VeryStrongPlus_days  UTCI_mean  UTCI_max
 1996  1.938017 102.545456 257.148773   4.367769       0.0       364.061981       261.516541             4.367769  33.216267 36.572517
 1997  1.822314 112.082642 244.417358   6.677686       0.0       363.177673       251.095047             6.677686  33.096584 36.720184
 1998  0.801653  74.582642 280.640503   8.975206       0.0       364.198334       289.615692             8.975206  33.840530 36.976440
 1999  2.194215 110.223137 248.256195   4.326446       0.0       362.805786       252.582642             4.326446  33.080292 36.558483
 2000  3.442149 106.735535 250.619827   5.202479       0.0       362.557861       255.822311             5.202479  33.127712 36.644981
 2001  1.826446 104.028923 253.429749   5.714876       0.0       363.173553       259.144623             5.714876  33.241592 36.660202
 2002  1.119835 103.780991 251.913223   8.185950       

## 11. Status akhir STEP 3

Jika semua pemeriksaan lolos, data siap digunakan untuk:
- peta spasial thermal stress;
- time series tahunan;
- trend Theil–Sen + Mann–Kendall;
- perbandingan heatwave Tmax vs UTCI;
- analisis apakah hari heatwave Tmax juga mengalami strong/very strong thermal stress.

In [12]:
# =========================================================
# 11. QC OUTPUT FINAL
# =========================================================

with xr.open_dataset(
    DAILY_CLASS_FILE,
    engine="netcdf4",
    decode_times=True,
    decode_timedelta=False
) as check_daily:

    n_daily = check_daily.sizes[
        "time"
    ]

with xr.open_dataset(
    ANNUAL_FILE,
    engine="netcdf4",
    decode_times=True,
    decode_timedelta=False
) as check_annual:

    n_year = check_annual.sizes[
        "year"
    ]

    year_min = int(
        check_annual["year"]
        .min()
        .values
    )

    year_max = int(
        check_annual["year"]
        .max()
        .values
    )


checks = {
    "daily_10958": (
        n_daily == 10958
    ),
    "annual_30_tahun": (
        n_year == 30
    ),
    "tahun_awal_1996": (
        year_min == 1996
    ),
    "tahun_akhir_2025": (
        year_max == 2025
    ),
    "kategori_consistent": (
        qc_fail == 0
    ),
}


print(
    "========== STATUS STEP 3 =========="
)

for name, status in checks.items():

    print(
        f"{name:22s}:",
        "LULUS"
        if status
        else "GAGAL"
    )


if all(
    checks.values()
):

    print(
        "\nUTCI THERMAL STRESS CLASSIFICATION: LULUS"
    )

    print(
        "Data siap masuk tahap plotting/trend "
        "dan integrasi dengan heatwave Tmax."
    )

else:

    raise RuntimeError(
        "STEP 3 belum lulus."
    )

========== STATUS STEP 3 ==========
daily_10958           : LULUS
annual_30_tahun       : LULUS
tahun_awal_1996       : LULUS
tahun_akhir_2025      : LULUS
kategori_consistent   : LULUS

UTCI THERMAL STRESS CLASSIFICATION: LULUS
Data siap masuk tahap plotting/trend dan integrasi dengan heatwave Tmax.


In [13]:
# =========================================================
# 12. BERSIHKAN MEMORI
# =========================================================

ds.close()

del values
del valid
del stress_class
del annual_vars

gc.collect()

print("Memori dibersihkan.")

Memori dibersihkan.
